In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
pip install --upgrade docling

# load docling converter & hyperidchunker

In [ ]:
from pathlib import Path
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions ,TableStructureOptions, TableFormerMode
from docling_core.transforms.chunker import HybridChunker
from docling_core.types.doc.labels import DocItemLabel
from docling_core.types.doc.document import ContentLayer
from docling_core.types.doc.base import ImageRefMode

## parsing

In [ ]:
# Add image_export_mode="referenced" to get image links instead of embedded
pipeline_options = PdfPipelineOptions(
    do_ocr=True,
    do_table_structure=True,
    generate_picture_images=True,
    image_export_mode="referenced", # <-- This gives you links, not embedded images
    image_scale=5,
    table_structure_options=TableStructureOptions(
        mode=TableFormerMode.ACCURATE, 
        do_cell_matching=False 
    )
)

converter = DocumentConverter(
    allowed_formats=[InputFormat.PDF], 
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

pdf_path = "/kaggle/input/datasets/ahmedezzattaha/m11-sm/m11_SM.pdf"
artifacts_output_dir = Path("extracted_artifacts")
artifacts_output_dir.mkdir(exist_ok=True)

print("//"*10 + "parsing operation started"+ "\\"*10)
result = converter.convert(pdf_path , page_range= (20,50))
doc = result.document
print("//"*10 + "parsing operation ended"+ "\\"*10)

## chunking

In [ ]:
# --- Chunking with HybridChunker (preserves image references) ---
chunker = HybridChunker(max_tokens=512) # Adjust max_tokens as needed
chunks = list(chunker.chunk(dl_doc=doc))

print(f"\nTotal chunks: {len(chunks)}")

### show sample of results to check

In [ ]:
for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(f"Text: {chunk.text[:30]}...")

    # Get contextualized version (includes headings for context)
    enriched = chunker.contextualize(chunk=chunk)
    print(f"Contextualized: {enriched[:300]}...")

    # Access image/provenance metadata per chunk
    for item in chunk.meta.doc_items:
        for prov in getattr(item, "prov", []):
            print(f" Page: {prov.page_no}, BBox: {prov.bbox}")

## saving the output 

In [ ]:
# save the prechunking doc as json.
doc.save_as_json(
    filename="output_V3_fullresult.json",
    artifacts_dir=artifacts_dir,
    image_mode=ImageRefMode.REFERENCED
)

# save the chunked doc as json 
with open("chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, indent=2, ensure_ascii=False)

print(f"Saved {len(ready_chunks)} chunks to chunks.json")


# save a protable version of parsing process result as zip file 
result.save(filename=output_dir / "conversion_result.zip")

# zip output to enable download

In [ ]:
# import shutil
# import os

# # Define the name of your output zip file
# output_filename = 'kaggle_output_100_131'
# # The directory you want to zip
# dir_to_zip = '/kaggle/working'

# # This creates 'my_kaggle_output.zip' in /kaggle/working
# shutil.make_archive(output_filename, 'zip', dir_to_zip)

# print(f"✅ Created {output_filename}.zip successfully!")

# clearing output

In [ ]:
# import os
# import shutil

# # Define the directory
# dir_path = '/kaggle/working'

# # Loop through the items in the directory
# for filename in os.listdir(dir_path):
#     file_path = os.path.join(dir_path, filename)
#     try:
#         if os.path.isfile(file_path) or os.path.islink(file_path):
#             os.unlink(file_path) # Deletes files or links
#         elif os.path.isdir(file_path):
#             shutil.rmtree(file_path) # Deletes subdirectories
#     except Exception as e:
#         print(f'Failed to delete {file_path}. Reason: {e}')

# print("Output directory cleared!")